# Stage 4b — Fair joint comparison on p(x, t, F, action status)
Literature models (M1, M3, M4, M5, M6a) + process-agnostic F submodel; structural process-anchored models (M6b, M6) model F as an encoding stage and (t−F) as the decision stage. Rapid-guess mean constrained ≤10 s. T4 GPU → Run all. Resumable.

In [ ]:
!pip -q install numpyro multipledispatch --no-deps
import os, time, json, numpy as np, pandas as pd, jax, jax.numpy as jnp, numpyro, numpyro.distributions as dist, optax, arviz as az
from numpyro.infer import SVI, Trace_ELBO, Predictive
from numpyro.infer.autoguide import AutoNormal
from google.colab import drive
drive.mount('/content/drive'); assert jax.devices()[0].platform=='gpu', 'Select T4 GPU'
S2='/content/drive/MyDrive/PISA2025_stage2'; S3='/content/drive/MyDrive/PISA2025_stage3'; S4='/content/drive/MyDrive/PISA2025_stage4'
OUT='/content/drive/MyDrive/PISA2025_stage4b'; os.makedirs(OUT,exist_ok=True)
CNTS=['ARM','KAZ','MDA','LVA','LTU','EST','JPN','ARE','SGP']; SEED=20251025; STEPS=20000; NDRAW=200; EPS=0.01; L10=np.log(10.)

D=pd.read_parquet(f'{S2}/sample_math.parquet')
ii,items=pd.factorize(D.item); pp,pers=pd.factorize(D.CNTSTUID)
pc=D.drop_duplicates('CNTSTUID').set_index('CNTSTUID').loc[pers]; cp=pd.Categorical(pc.CNT,categories=CNTS).codes
ts=D.t.values/10.; fs=D.f.values/10.; hasf=~np.isnan(fs); fs=np.where(hasf,np.minimum(fs,ts),0.)
dec=np.where(hasf,ts-fs,np.nan); mindec=pd.Series(dec).groupby(ii).min().fillna(0).values
r=np.clip(np.where(hasf,fs/ts,0.5),1e-4,1-1e-4)
esc=pc.ESCS.fillna(pc.groupby('CNT').ESCS.transform('mean')).fillna(0).values; fem=(pc.ST004D01T.values==1).astype(np.float32)
ai=pc.AIUSESCH.values; aiobs=~np.isnan(ai); g=pd.Series(ts).groupby(ii)
d=dict(ii=jnp.array(ii),pp=jnp.array(pp),cp=jnp.array(cp),x=jnp.array(D.x.values,jnp.float32),ts=jnp.array(ts,jnp.float32),lts=jnp.array(np.log(D.t.values),jnp.float32),
       mint=jnp.array(g.min().values,jnp.float32),tmax=jnp.array(g.max().values,jnp.float32),mindec=jnp.array(mindec,jnp.float32),
       fs=jnp.array(fs,jnp.float32),hasf=jnp.array(hasf),noact=jnp.array(~hasf,jnp.float32),r=jnp.array(r,jnp.float32),
       ai=jnp.array(np.nan_to_num(ai),jnp.float32),aiobs=jnp.array(aiobs),esc=jnp.array(esc,jnp.float32),fem=jnp.array(fem))
nI,nP,nC=len(items),len(pers),len(CNTS); print(nI,nP,len(D),'has F',hasf.mean().round(3))

def wfpt(t,v,a,w=0.5):
    u=t/a**2; k=jnp.arange(1,21.)
    sl=jnp.sum(k*jnp.sin(k*jnp.pi*w)*jnp.exp(-(k**2-1)*jnp.pi**2*u[...,None]/2),-1)
    large=jnp.log(jnp.pi)+jnp.log(jnp.maximum(sl,1e-20))-jnp.pi**2*u/2
    ww=w+2*jnp.arange(-5,6.); ss=jnp.sum(ww*jnp.exp(-(ww**2-w**2)/(2*u[...,None])),-1)
    small=-0.5*jnp.log(2*jnp.pi*u**3)+jnp.log(jnp.maximum(ss,1e-20))-w**2/(2*u)
    return jnp.where(u<0.5,small,large)-2*jnp.log(a)-v*a*w-v**2*t/2
def wsurv(t,v,a):
    k=jnp.arange(1,151.); sk=k*jnp.sin(k*jnp.pi/2); lam=v[...,None]**2/2+k**2*jnp.pi**2/(2*a[...,None]**2)
    s=jnp.sum(sk/lam*jnp.exp(-(lam-lam[...,:1])*t[...,None]),-1)
    ls=jnp.log(2*jnp.pi/a**2)+(jnp.abs(v*a/2)+jnp.log1p(jnp.exp(-jnp.abs(v*a)))-jnp.log(2.))+jnp.log(jnp.maximum(s,1e-30))-lam[...,0]*t
    return jnp.where(t/a**2<0.01,0.,jnp.minimum(ls,0.))

# ---------- (A) process-agnostic F submodel: P(no action)_i and Beta(F/t)_i ; added to every literature model ----------
def LL_gen(p,d):
    i=d['ii']; ls=dist.Bernoulli(logits=p['nu'][i]).log_prob(d['noact'])
    lf=dist.Beta(jnp.exp(p['la_b'][i]),jnp.exp(p['lb_b'][i])).log_prob(d['r'])-d['lts']      # density of F in seconds
    return ls+jnp.where(d['hasf'],lf,0.)
def gen_model(d):
    p={'nu':numpyro.sample('nu',dist.Normal(-2.,2.).expand([nI])),'la_b':numpyro.sample('la_b',dist.Normal(0.,1.).expand([nI])),'lb_b':numpyro.sample('lb_b',dist.Normal(0.,1.).expand([nI]))}
    numpyro.factor('obs',LL_gen(p,d).sum())

# ---------- (B) structural process-anchored model: encoding stage F ~ LogNormal, decision stage (t-F) ~ diffusion/persistence race ----------
K5=5; PN=['drift','log_caution','persistence','disengagement','encoding_speed']
def eta_of(p,d,name):
    mu=jnp.concatenate([jnp.zeros((1,K5)),p['mu_free']],0)[d['cp']]
    if name=='M6_full':
        aif=jnp.where(d['aiobs'],d['ai'],p['mai'][d['cp']]+p['sai']*p['ai_z'])
        mu=mu+jnp.stack([aif,d['esc'],d['fem']],1)@p['Gam'].T
    return mu+p['z']@(p['sd'][:,None]*p['Lcorr']).T
def LL_str(p,d,name):
    e=eta_of(p,d,name); i,q=d['ii'],d['pp']; t=d['ts']; hf=d['hasf']
    v=jnp.clip(e[q,0]-p['b'][i],-8.,8.); a=jnp.exp(jnp.clip(p['la'][i]+e[q,1],-3.,4.))
    td=jnp.where(hf,jnp.maximum(t-d['fs']-d['mindec'][i]*jax.nn.sigmoid(p['r0'][i])*0.99,1e-4),
                    jnp.maximum(t-d['mint'][i]*jax.nn.sigmoid(p['r'][i])*0.99,1e-4))
    lw=jnp.where(d['x']>0.5,wfpt(td,-v,a),wfpt(td,v,a))
    zc=(d['lts']-L10-e[q,2]-p['oc'][i])/p['sc']; lfc=dist.Normal(0.,1.).log_prob(zc)-jnp.log(p['sc'])-(d['lts']-L10); lSc=jax.scipy.special.log_ndtr(-zc)
    lg=dist.Bernoulli(probs=p['g']).log_prob(d['x'])
    race=jnp.logaddexp(lw+lSc,lfc+wsurv(td,v,a)+lg)
    race=jnp.logaddexp(jnp.log(1-EPS)+race,jnp.log(EPS)+jnp.log(0.5)-jnp.log(d['tmax'][i]))-L10            # (x,t) in seconds
    lenc=dist.LogNormal(p['eps'][i]-e[q,4],p['se']).log_prob(jnp.maximum(d['fs'],1e-4))-L10                # F in seconds
    fgen=dist.Beta(jnp.exp(p['la_b'][i]),jnp.exp(p['lb_b'][i])).log_prob(d['r'])-d['lts']
    mur=-jax.nn.softplus(p['mur_raw']); sr=0.1+0.9*jax.nn.sigmoid(p['sr_raw'])                              # rapid mean <= 10 s
    rg=lg+dist.LogNormal(mur,sr).log_prob(t)-L10
    k=p['kap0']+e[q,3]
    mix=jnp.logaddexp(jax.nn.log_sigmoid(-k)+race+jnp.where(hf,lenc,0.),jax.nn.log_sigmoid(k)+rg+jnp.where(hf,fgen,0.))
    lnu=p['nu'][i]+(p['lam_na']*e[q,3] if name=='M6_full' else 0.)
    return mix+dist.Bernoulli(logits=lnu).log_prob(d['noact'])
def str_model(name):
    def model(d):
        S=numpyro.sample; p={}
        p['mu_free']=S('mu_free',dist.Normal(0.,1.).expand([nC-1,K5])); p['Lcorr']=S('Lcorr',dist.LKJCholesky(K5,2.))
        p['sd']=S('sd',dist.HalfNormal(1.).expand([K5])); p['z']=S('z',dist.Normal(0.,1.).expand([nP,K5]))
        for k,pr in [('b',dist.Normal(0.,2.)),('la',dist.Normal(1.,1.)),('r',dist.Normal(0.,1.5)),('r0',dist.Normal(-1.,1.5)),('oc',dist.Normal(1.5,1.)),
                     ('eps',dist.Normal(1.,1.)),('nu',dist.Normal(-2.,2.)),('la_b',dist.Normal(0.,1.)),('lb_b',dist.Normal(0.,1.))]: p[k]=S(k,pr.expand([nI]))
        p['sc']=S('sc',dist.HalfNormal(1.)); p['se']=S('se',dist.HalfNormal(1.)); p['g']=S('g',dist.Beta(2.,6.))
        p['kap0']=S('kap0',dist.Normal(-3.,1.)); p['mur_raw']=S('mur_raw',dist.Normal(1.,1.)); p['sr_raw']=S('sr_raw',dist.Normal(0.,1.))
        if name=='M6_full':
            p['lam_na']=S('lam_na',dist.Normal(0.,2.)); p['Gam']=S('Gam',dist.Normal(0.,1.).expand([K5,3]))
            p['mai']=S('mai',dist.Normal(0.,1.).expand([nC])); p['sai']=S('sai',dist.HalfNormal(1.)); p['ai_z']=S('ai_z',dist.Normal(0.,1.).expand([nP]))
            p['zeta']=S('zeta',dist.Normal(0.,2.).expand([nC])); p['zw']=S('zw',dist.Normal(0.,1.).expand([2]))
            numpyro.factor('ai_obs',jnp.where(d['aiobs'],dist.Normal(p['mai'][d['cp']],p['sai']).log_prob(d['ai']),0.).sum())
            e=eta_of(p,d,name); numpyro.factor('ai_miss',dist.Bernoulli(logits=p['zeta'][d['cp']]+p['zw'][0]*e[:,2]+p['zw'][1]*e[:,0]).log_prob((~d['aiobs']).astype(jnp.float32)).sum())
        numpyro.factor('obs',LL_str(p,d,name).sum())
    return model

def fit(model,ll_fn,tag,lr=0.005):
    guide=AutoNormal(model,init_scale=0.05)
    opt=numpyro.optim.optax_to_numpyro(optax.chain(optax.clip_by_global_norm(10.),optax.adam(optax.exponential_decay(lr,5000,0.5,end_value=lr/20))))
    t0=time.time(); res=SVI(model,guide,opt,Trace_ELBO()).run(jax.random.PRNGKey(SEED),STEPS,d,progress_bar=True,stable_update=True); rt=time.time()-t0
    Ls=np.asarray(res.losses); print(tag,'finite ELBO share',np.isfinite(Ls).mean()); assert np.isfinite(Ls[-1000:]).mean()>0.9
    s=Predictive(guide,params=res.params,num_samples=NDRAW)(jax.random.PRNGKey(1),d); s={k:v for k,v in s.items() if k not in('obs','ai_obs','ai_miss')}
    pick=lambda j: jax.tree_util.tree_map(lambda a:a[j],s)
    ll=np.asarray(jax.lax.map(lambda j: ll_fn(pick(j),d),jnp.arange(NDRAW))); np.save(f'{OUT}/{tag}_ll.npy',ll.astype(np.float32))
    return s,pick,Ls,rt,ll

def loo_of(ll): return az.loo(az.from_dict(posterior={'dummy':np.zeros((1,ll.shape[0]))},log_likelihood={'obs':ll[None]}))
RES={}
if not os.path.exists(f'{OUT}/GEN_ll.npy'): fit(gen_model,LL_gen,'GEN',lr=0.02)
llg=np.load(f'{OUT}/GEN_ll.npy')
for m,src in [('M1_vdl',S3),('M3_mix',S3),('M4_diff',S3),('M5_race',S4),('M6a_mix',S4)]:
    ll=az.from_netcdf(f'{src}/{m}_ll.nc').log_likelihood['obs'].values[0]+llg; lo=loo_of(ll)
    RES[m+'+Fgen']={'elpd_loo':round(float(lo.elpd_loo),1),'se':round(float(lo.se),1),'p_loo':round(float(lo.p_loo),1)}; np.save(f'{OUT}/{m}_joint_ll.npy',ll.astype(np.float32))
for name in ['M6b_anchor','M6_full']:
    if os.path.exists(f'{OUT}/{name}_res.json'): RES[name]=json.load(open(f'{OUT}/{name}_res.json'))['loo']; continue
    s,pick,Ls,rt,ll=fit(str_model(name),lambda p,dd,nm=name: LL_str(p,dd,nm),name); lo=loo_of(ll)
    E=np.asarray(jax.lax.map(lambda j: eta_of(pick(j),d,name),jnp.arange(NDRAW)))
    P=pc[['CNT','AIUSESCH','ESCS','ST004D01T','EFFORT1','W_FSTUWT','ST438Q01DA','ST438Q02DA','ST438Q03DA','ST438Q04DA']].copy()
    for k in range(K5): P[PN[k]]=E[...,k].mean(0); P[PN[k]+'_sd']=E[...,k].std(0)
    P.reset_index().to_csv(f'{OUT}/{name}_persons.csv',index=False)
    q=lambda k: np.asarray(s[k]); loo={'elpd_loo':round(float(lo.elpd_loo),1),'se':round(float(lo.se),1),'p_loo':round(float(lo.p_loo),1),'pareto_k_gt_0.7_pct':round(100*float((lo.pareto_k.values>0.7).mean()),2)}
    out={'loo':loo,'runtime_min':round(rt/60,1),'elbo_rel_change':round(float((np.nanmean(Ls[-1500:-1000])-np.nanmean(Ls[-500:]))/abs(np.nanmean(Ls[-500:]))),6),
         'g':round(float(q('g').mean()),3),'rapid_mean_sec':round(float(10*np.exp(-np.log1p(np.exp(q('mur_raw'))).mean())),2),'p_disengaged_mean':round(float(jax.nn.sigmoid(q('kap0')).mean()),3),
         'corr':np.round(np.einsum('nij,nkj->nik',q('Lcorr'),q('Lcorr')).mean(0),2).tolist()}
    if name=='M6_full':
        G=q('Gam'); out['lam_noact']=round(float(q('lam_na').mean()),3); out['zeta_persist_drift']=q('zw').mean(0).round(3).tolist()
        out['AI_effect']={PN[k]:{'mean':round(float(G[:,k,0].mean()),4),'q2.5':round(float(np.quantile(G[:,k,0],.025)),4),'q97.5':round(float(np.quantile(G[:,k,0],.975)),4)} for k in range(K5)}
    json.dump(out,open(f'{OUT}/{name}_res.json','w'),indent=1); print(name,json.dumps(out)); RES[name]=loo
mk=lambda f: az.from_dict(posterior={'dummy':np.zeros((1,NDRAW))},log_likelihood={'obs':np.load(f)[None]})
ids={m+'+Fgen':mk(f'{OUT}/{m}_joint_ll.npy') for m in ['M1_vdl','M3_mix','M4_diff','M5_race','M6a_mix']}
ids.update({m:mk(f'{OUT}/{m}_ll.npy') for m in ['M6b_anchor','M6_full']})
cmp=az.compare(ids,ic='loo'); cmp.to_csv(f'{OUT}/stage4b_joint_compare.csv'); print(cmp[['rank','elpd_loo','p_loo','elpd_diff','dse']].round(1)); print('STAGE4b DONE')
